# VietHandOCR Part 3: Digital Image Processing (DIP) Pipeline

Welcome to **Part 2** of the VietHandOCR pipeline.
- **Previous Notebook**: [Part 2: Baseline Evaluation](./02_Baseline_Evaluation.ipynb)
- **Next Notebook**: [Part 4: VietOCR Model Training](./04_VietOCR_Training.ipynb)

## Introduction
Based on Gonzalez's "Digital Image Processing, 4th Edition", we implement:
1. Illumination Correction (CLAHE)
2. Binarization & Noise Reduction (Adaptive Thresholding, Morphological Ops)
3. Skew Detection & Correction (Canny + Hough Lines)
4. Paragraph Segmentation (Horizontal Projection Profiles)


In [ ]:
import cv2, gc
import numpy as np
import matplotlib.pyplot as plt

def preprocess_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(img)
    binary = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return img, cleaned


In [ ]:
def deskew_image(img):
    edges = cv2.Canny(img, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)
    if lines is None: return img
    angles = [np.degrees(np.arctan2(l[0][3]-l[0][1], l[0][2]-l[0][0])) for l in lines if -45 < np.degrees(np.arctan2(l[0][3]-l[0][1], l[0][2]-l[0][0])) < 45]
    if not angles: return img
    median_angle = np.median(angles)
    (h, w) = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), median_angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


In [ ]:
def segment_paragraph(binary_img):
    horizontal_projection = np.sum(binary_img, axis=1)
    line_regions = horizontal_projection > (np.max(horizontal_projection) * 0.1)
    changes = np.diff(line_regions.astype(int))
    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]
    if len(starts) == 0: return []
    if line_regions[0]: starts = np.insert(starts, 0, 0)
    if line_regions[-1]: ends = np.append(ends, len(line_regions)-1)
    lines = [binary_img[max(0, s-5):min(binary_img.shape[0], e+5), :] for s, e in zip(starts, ends) if e - s > 15]
    return lines


In [ ]:
import os

def full_dip_pipeline(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    # 1. Illumination Correction
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(img)
    # 2. Adaptive Binarization
    binary = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    # 3. Morphological Ops
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    # 4. Skew Correction
    deskewed = deskew_image(cleaned)
    return deskewed

def process_and_save_dataset(txt_file, output_txt, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    if not os.path.exists(txt_file): return
    with open(txt_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    with open(output_txt, 'w', encoding='utf-8') as f:
        for line in lines:
            parts = line.strip().split('\t')
            if len(parts) != 2: continue
            img_path, label = parts
            
            processed = full_dip_pipeline(img_path)
            if processed is not None:
                base_name = os.path.basename(img_path)
                out_path = os.path.join(out_dir, base_name)
                cv2.imwrite(out_path, processed)
                f.write(f"{out_path}\t{label}\n")

process_and_save_dataset('train.txt', 'processed_train.txt', 'Processed_Datasets/train')
process_and_save_dataset('val.txt', 'processed_val.txt', 'Processed_Datasets/val')
process_and_save_dataset('test.txt', 'processed_test.txt', 'Processed_Datasets/test')